# 07 · Evaluación única en test

Evalúa una sola vez el seasonal naive y el modelo congelado en los 13 orígenes, sin reentrenar. Repetir con el mismo `model_id` detiene el proceso (SC-007).

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

In [ ]:
from src.forecasting import run_test_evaluation_stage

evaluation = run_test_evaluation_stage(cfg)
evaluation

Los parámetros usados son los del archivo congelado: el hash registrado en el ledger coincide con el manifiesto.

In [ ]:
from src.freeze import TestLedger, load_manifest

arts = load_manifest()["artifacts"]
for e in TestLedger().entries():
    name = "baseline_models" if e["model_id"].startswith("snaive") else "frozen_models"
    assert e["frozen_models_sha256"] == arts[name]["sha256"]
    print(e["model_id"], "→ hash verificado")

In [ ]:
chosen = evaluation.iloc[1]
if chosen["beats_snaive"]:
    print(f"Modelo aceptado: WRMSSE {chosen['wrmsse']:.4f} < seasonal naive {evaluation.iloc[0]['wrmsse']:.4f}")
else:
    print("El modelo NO supera al seasonal naive: el RMV no se cierra. No se cambian parámetros (Principio VI).")